In [3]:
import pandas as pd
import re

In [4]:
data = pd.read_csv('krisha_raw.csv')

data.head()

,title,price,address,description
0,4-комнатная квартира · 121.7 м² · 2 этаж,109 500 000 〒,"Медеуский р-н, Сагадат Нурмагамбетов 71","жил. комплекс RAMS Garden, монолитный дом, 202..."
1,3-комнатная квартира · 72.1 м² · 6/8 этаж,62 000 000 〒,"Бостандыкский р-н, мкр Алмагуль, Жарокова","панельный дом, 1988 г.п., состояние: требует р..."
2,4-комнатная квартира · 83 м² · 5/5 этаж,62 000 000 〒,"Бостандыкский р-н, Жандосова 31 Б — Пересечени...","панельный дом, 1989 г.п., состояние: не новый,..."
3,3-комнатная квартира · 97 м² · 5/5 этаж,126 100 000 〒,"Медеуский р-н, мкр Коктобе, Сагадат Нурмагамбе...","жил. комплекс Koktobe city, монолитный дом, 20..."
4,2-комнатная квартира · 77 м² · 8/10 этаж,38 500 000 〒,"Наурызбайский р-н, мкр Шугыла, Алтын-орда","жил. комплекс Alma City 5, монолитный дом, 202..."


In [5]:
# комнаты из title
data['rooms'] = data['title'].str.extract(r'(\d+)-комнатная')

# площадь из title
data['area'] = data['title'].str.extract(r'(\d+\.?\d*)\s*м²')

# этаж и этажность из title
data['floor'] = data['title'].str.extract(r'(\d+)/\d+\s*этаж')
data['total_floors'] = data['title'].str.extract(r'\d+/(\d+)\s*этаж')

# убрал пробелы и символы из цены
data['price'] = data['price'].str.replace(r'[^\d]', '', regex=True)

# год постройки из description
data['year'] = data['description'].str.extract(r'(\d{4})\s*г\.п\.')

# тип дома из description
data['building_type'] = data['description'].str.extract(r'(монолитный|кирпичный|панельный)\s*дом')

# Перевод в числа
data['rooms'] = pd.to_numeric(data['rooms'])
data['area'] = pd.to_numeric(data['area'])
data['floor'] = pd.to_numeric(data['floor'])
data['total_floors'] = pd.to_numeric(data['total_floors'])
data['price'] = pd.to_numeric(data['price'])
data['year'] = pd.to_numeric(data['year'])


# сброс строк где нет цены или площади
data = data.dropna(subset=['price', 'area'])


# сброс ненужных признаков
data_clean = data.drop(columns=['title', 'description', 'address'])

data_clean = pd.get_dummies(data_clean, columns=['building_type'], dtype=int)

print(data_clean.head())

       price  rooms   area  floor  total_floors    year  \
0  109500000      4  121.7    NaN           NaN  2026.0   
1   62000000      3   72.1    6.0           8.0  1988.0   
2   62000000      4   83.0    5.0           5.0  1989.0   
3  126100000      3   97.0    5.0           5.0  2019.0   
4   38500000      2   77.0    8.0          10.0  2022.0   

   building_type_кирпичный  building_type_монолитный  building_type_панельный  
0                        0                         1                        0  
1                        0                         0                        1  
2                        0                         0                        1  
3                        0                         1                        0  
4                        0                         1                        0  


In [6]:
# сброс строки nan и выбросов
data_clean = data_clean.dropna(subset=['price', 'area', 'rooms'])
data_clean = data_clean[(data_clean['price'] >= 5000000) & (data_clean['price'] <= 500000000)]
data_clean = data_clean[(data_clean['area'] >= 15) & (data_clean['area'] <= 500)]

data_clean.head()

,price,rooms,area,floor,total_floors,year,building_type_кирпичный,building_type_монолитный,building_type_панельный
0,109500000,4,121.7,NaN,NaN,2026.0,0,1,0
1,62000000,3,72.1,6.0,8.0,1988.0,0,0,1
2,62000000,4,83.0,5.0,5.0,1989.0,0,0,1
3,126100000,3,97.0,5.0,5.0,2019.0,0,1,0
4,38500000,2,77.0,8.0,10.0,2022.0,0,1,0


In [7]:
print(data_clean.isnull().sum())

# сброс строк с пропусками
data_clean = data_clean.dropna()

print(f"Размер после удаления пропусков: {data_clean.shape}")

price                         0
rooms                         0
area                          0
floor                       459
total_floors                459
year                         21
building_type_кирпичный       0
building_type_монолитный      0
building_type_панельный       0
dtype: int64
Размер после удаления пропусков: (9463, 9)


In [8]:
# Сохранение
data_clean.to_csv('krisha_clean.csv', index=False, encoding='utf-8-sig')

print(f"Чистый датасет: {data_clean.shape}")
print(data_clean.head())
print(data_clean.isnull().sum())

Чистый датасет: (9463, 9)
       price  rooms  area  floor  total_floors    year  \
1   62000000      3  72.1    6.0           8.0  1988.0   
2   62000000      4  83.0    5.0           5.0  1989.0   
3  126100000      3  97.0    5.0           5.0  2019.0   
4   38500000      2  77.0    8.0          10.0  2022.0   
5   51000000      2  77.0    3.0          10.0  2021.0   

   building_type_кирпичный  building_type_монолитный  building_type_панельный  
1                        0                         0                        1  
2                        0                         0                        1  
3                        0                         1                        0  
4                        0                         1                        0  
5                        0                         1                        0  
price                       0
rooms                       0
area                        0
floor                       0
total_floors             

(9463, 9)